In [4]:
import json
from collections import Counter
from itertools import product

import pandas as pd


# =========================
# 1. LOAD DATA
# =========================
json_path = "../metadata_sample/splits_house/all_with_split.json"

with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# Pastikan data berupa list of dict
if isinstance(data, dict):
    # kalau ternyata file berisi dict dengan key tertentu
    # sesuaikan key ini bila perlu
    if "data" in data:
        data = data["data"]
    else:
        raise ValueError("Format JSON tidak dikenali. Expected list of records or dict with key 'data'.")


# =========================
# 2. CONVERT TO DATAFRAME
# =========================
rows = []

for item in data:
    actual_label = item.get("actual_label", {})
    rows.append({
        "house_id": item.get("house_id"),
        "no_kk": item.get("no_kk"),
        "split": item.get("split"),
        "house_type": item.get("house_type"),
        "actual_atap": actual_label.get("atap"),
        "actual_dinding": actual_label.get("dinding"),
        "actual_lantai": actual_label.get("lantai"),
    })

df = pd.DataFrame(rows)

# Cek data awal
print("Jumlah baris:", len(df))
print("Kolom:", df.columns.tolist())
print("\nSample data:")
print(df.head())


# =========================
# 3. BASIC EDA
# =========================
print("\n=== Distribusi split ===")
print(df["split"].value_counts(dropna=False))

print("\n=== Distribusi house_type ===")
print(df["house_type"].value_counts(dropna=False))

print("\n=== Distribusi actual_atap ===")
print(df["actual_atap"].value_counts(dropna=False))

print("\n=== Distribusi actual_dinding ===")
print(df["actual_dinding"].value_counts(dropna=False))

print("\n=== Distribusi actual_lantai ===")
print(df["actual_lantai"].value_counts(dropna=False))


# =========================
# 4. COUNT KOMBINASI YANG DIMINTA
# =========================
combo_cols = [
    "split",
    "house_type",
    "actual_atap",
    "actual_dinding",
    "actual_lantai",
]

combo_counts = (
    df.groupby(combo_cols, dropna=False)
      .size()
      .reset_index(name="count")
      .sort_values("count", ascending=False)
      .reset_index(drop=True)
)

print("\n=== Kombinasi split | house_type | actual_atap | actual_dinding | actual_lantai ===")
print(combo_counts.to_string(index=False))


# =========================
# 5. PIVOT TABLE UNTUK ANALISIS LEBIH MUDAH
# =========================

# 5a. Kombinasi per split vs house_type
pivot_split_house = pd.pivot_table(
    df,
    index="split",
    columns="house_type",
    values="house_id",
    aggfunc="count",
    fill_value=0
)

print("\n=== Pivot: split x house_type ===")
print(pivot_split_house)

# 5b. Kombinasi split vs label atap
pivot_split_atap = pd.pivot_table(
    df,
    index="split",
    columns="actual_atap",
    values="house_id",
    aggfunc="count",
    fill_value=0
)

print("\n=== Pivot: split x actual_atap ===")
print(pivot_split_atap)

# 5c. Kombinasi split vs label dinding
pivot_split_dinding = pd.pivot_table(
    df,
    index="split",
    columns="actual_dinding",
    values="house_id",
    aggfunc="count",
    fill_value=0
)

print("\n=== Pivot: split x actual_dinding ===")
print(pivot_split_dinding)

# 5d. Kombinasi split vs label lantai
pivot_split_lantai = pd.pivot_table(
    df,
    index="split",
    columns="actual_lantai",
    values="house_id",
    aggfunc="count",
    fill_value=0
)

print("\n=== Pivot: split x actual_lantai ===")
print(pivot_split_lantai)


# =========================
# 6. OPSIONAL: EDA LEBIH DETAIL PER SPLIT
# =========================
for s in ["train", "val", "test"]:
    sub = df[df["split"] == s]
    print(f"\n=== Detail untuk split: {s} ===")
    print("Jumlah rumah:", len(sub))
    print("House type:")
    print(sub["house_type"].value_counts(dropna=False))
    print("Atap:")
    print(sub["actual_atap"].value_counts(dropna=False))
    print("Dinding:")
    print(sub["actual_dinding"].value_counts(dropna=False))
    print("Lantai:")
    print(sub["actual_lantai"].value_counts(dropna=False))


# =========================
# 7. OPSIONAL: CEK KOMBINASI YANG TIDAK ADA
# =========================
all_splits = ["train", "val", "test"]
all_house_types = ["multi", "single_exterior_only", "single_interior_only"]

# Ambil label unik dari data
all_atap = sorted(df["actual_atap"].dropna().unique().tolist())
all_dinding = sorted(df["actual_dinding"].dropna().unique().tolist())
all_lantai = sorted(df["actual_lantai"].dropna().unique().tolist())

all_possible_combos = list(product(all_splits, all_house_types, all_atap, all_dinding, all_lantai))
existing_combos = set(tuple(x) for x in df[combo_cols].itertuples(index=False, name=None))

missing_combos = [c for c in all_possible_combos if c not in existing_combos]

print("\n=== Jumlah kombinasi mungkin ===")
print("Total possible combos :", len(all_possible_combos))
print("Existing combos       :", len(existing_combos))
print("Missing combos        :", len(missing_combos))

# Simpan yang kosong juga kalau ingin dianalisis
missing_df = pd.DataFrame(
    missing_combos,
    columns=combo_cols
)
missing_df["count"] = 0

Jumlah baris: 5858
Kolom: ['house_id', 'no_kk', 'split', 'house_type', 'actual_atap', 'actual_dinding', 'actual_lantai']

Sample data:
  house_id                                              no_kk  split  \
0   H61592               FAM_87585d6db728401b84e8cab0d2b77460  train   
1   H61598               FAM_fbcf9231dfd94e9fbed47f19f07f81ad  train   
2   H43054  FAM_85e47c29ba6704725e4b8b56c475d2c95171758e21...  train   
3   H42327  FAM_c49113dee19451d5516d51834dcffb6fd58c6dd9b4...  train   
4   H43247  FAM_9786643ad2526cbe3b9e1d40b4f5b96dd0a76c8863...  train   

  house_type       actual_atap                    actual_dinding  \
0      multi           Genteng  Kayu/papan/gypsum/GRC/calciboard   
1      multi           Genteng                            Tembok   
2      multi             Beton                            Tembok   
3      multi  Tidak terdeteksi  Kayu/papan/gypsum/GRC/calciboard   
4      multi  Tidak terdeteksi                            Tembok   

         actual_lantai 

In [5]:
combo_table = (
    df.groupby([
        "split",
        "house_type",
        "actual_atap",
        "actual_dinding",
        "actual_lantai"
    ])
    .size()
    .reset_index(name="count")
    .sort_values(
        ["split", "house_type", "count"],
        ascending=[True, True, False]
    )
)

display(combo_table)

,split,house_type,actual_atap,actual_dinding,actual_lantai,count
5,test,multi,Genteng,Tembok,Keramik,250
8,test,multi,Tidak terdeteksi,Tembok,Keramik,211
6,test,multi,Genteng,Tembok,Semen/bata merah,37
0,test,multi,Asbes,Kayu/papan/gypsum/GRC/calciboard,Keramik,1
1,test,multi,Asbes,Tembok,Marmer/granit,1
...,...,...,...,...,...,...
172,val,single_exterior_only,Kayu/sirap,Batang kayu,Tidak terdeteksi,1
173,val,single_exterior_only,Seng,Anyaman bambu,Tidak terdeteksi,1
174,val,single_exterior_only,Seng,Kayu/papan/gypsum/GRC/calciboard,Tidak terdeteksi,1
175,val,single_exterior_only,Tidak terdeteksi,Kayu/papan/gypsum/GRC/calciboard,Tidak terdeteksi,1


In [8]:
# ====================================
# TABEL KOMBINASI TRAIN/VAL/TEST
# ====================================

combo_pivot = (
    df.groupby([
        "house_type",
        "actual_atap",
        "actual_dinding",
        "actual_lantai",
        "split"
    ])
    .size()
    .unstack(fill_value=0)
)

# pastikan semua kolom split ada
for col in ["train", "val", "test"]:
    if col not in combo_pivot.columns:
        combo_pivot[col] = 0

# Urutkan kolom
combo_pivot = combo_pivot[["train", "val", "test"]]

combo_pivot = combo_pivot.reset_index()

# total per kombinasi
combo_pivot["total"] = (
    combo_pivot["train"]
    + combo_pivot["val"]
    + combo_pivot["test"]
)

# urutkan dari kombinasi terbanyak
combo_pivot = combo_pivot.sort_values(
    "total",
    ascending=False
)

display(combo_pivot.head(20))

split,house_type,actual_atap,actual_dinding,actual_lantai,train,val,test,total
44,multi,Genteng,Tembok,Keramik,211,250,250,711
86,multi,Tidak terdeteksi,Tembok,Keramik,251,210,211,672
129,single_interior_only,Tidak terdeteksi,Tidak terdeteksi,Keramik,297,186,186,669
108,single_exterior_only,Genteng,Tembok,Tidak terdeteksi,150,147,148,445
48,multi,Genteng,Tembok,Semen/bata merah,347,33,37,417
126,single_exterior_only,Tidak terdeteksi,Tembok,Tidak terdeteksi,285,32,35,352
133,single_interior_only,Tidak terdeteksi,Tidak terdeteksi,Semen/bata merah,345,0,0,345
13,multi,Asbes,Tembok,Keramik,240,0,0,240
90,multi,Tidak terdeteksi,Tembok,Semen/bata merah,213,0,0,213
98,single_exterior_only,Asbes,Tembok,Tidak terdeteksi,178,0,0,178


In [9]:
output_csv = "../metadata_sample/splits_house/combo_distribution.csv"

combo_pivot.to_csv(
    output_csv,
    index=False,
    encoding="utf-8-sig"
)

print(f"Saved: {output_csv}")

Saved: ../metadata_sample/splits_house/combo_distribution.csv
